# ARES: state-augmented SAITS + LiDAR BarrierNet
The outer run split is 56/12/12. Twelve of the 56 training runs are reserved for empirical error calibration, leaving 44 gradient-fitting runs. The online controller receives motion state and measured/reconstructed LiDAR, never the obstacle map. Full assumptions and limits: [`docs/AUGMENTED_SAITS_BARRIER_COLAB.md`](../docs/AUGMENTED_SAITS_BARRIER_COLAB.md) and [`docs/LIDAR_BARRIER_DERIVATION.md`](../docs/LIDAR_BARRIER_DERIVATION.md). No completed T4 training score, speed, or safety guarantee is claimed.


## 1. Locate the existing checkout


In [1]:
from pathlib import Path
import json, sys, os, subprocess, platform, shutil
from IPython.display import display, FileLink

REPO_ROOT = Path(r"A:\Projects\ARES").expanduser().resolve()
PYTHON = Path(sys.executable).resolve()
BATCH_ID = "c3eaab07-d882-4825-98a5-84c278cf8935"
CALIBRATION = REPO_ROOT / "data/calibration/saits_barrier_dynamics/de5533e2-d348-4e43-a06f-4e5bb2e50055/calibration.json"
BASE_CONFIG = REPO_ROOT / "configs/augmented_saits_barrier.json"
required = [REPO_ROOT / name for name in (
    "scripts/train_augmented_saits_barrier.py", "scripts/validate_dynamics_calibration.py",
    "scripts/export_augmented_saits_barrier.py", "scripts/ares_safe_training/pipeline.py",
    f"data/batches/{BATCH_ID}/python_index.csv")]
required += [CALIBRATION, CALIBRATION.with_name("fixtures.mat"), BASE_CONFIG]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing local project inputs: " + repr(missing))
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Select your existing .venv kernel in VS Code before continuing.")
scripts_path = str(REPO_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
# Also check that the active kernel can launch child processes, as training uses one.
probe = subprocess.run([str(PYTHON), "-c", "import sys; print(sys.executable)"],
                       check=True, capture_output=True, text=True)
print("Notebook kernel:", PYTHON)
print("Child interpreter:", probe.stdout.strip())
print("Platform:", platform.platform())
print("Checkout:", REPO_ROOT)
print("Local setup starts here. The two introductory cells above retain their original Colab wording.")


Notebook kernel: A:\Projects\ARES\.venv\Scripts\python.exe
Child interpreter: A:\Projects\ARES\.venv\Scripts\python.exe
Platform: Windows-10-10.0.26200-SP0
Checkout: A:\Projects\ARES
Local setup starts here. The two introductory cells above retain their original Colab wording.


## 2. Check the existing local environment

The next cell checks your installed packages without importing PyTorch. If your environment has CPU-only PyTorch, use the optional CUDA repair cell immediately afterward. After changing PyTorch, restart the kernel and run again from cell 3.


In [2]:
import importlib.util
from importlib.metadata import version as package_version

required_packages = ("numpy", "pandas", "scipy", "torch")
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    install_command = [str(PYTHON), "-m", "pip", "install", *missing_packages]
    raise RuntimeError("Missing packages in this kernel: " + repr(missing_packages)
                       + ". Install into this environment, then restart the kernel. "
                       + "Use the CUDA repair cell for torch rather than a default CPU installation. "
                       + subprocess.list2cmdline(install_command))
print({name: package_version(name) for name in required_packages})
print("PyTorch has not been imported by this setup cell.")


{'numpy': '2.4.6', 'pandas': '3.1.1', 'scipy': '1.17.1', 'torch': '2.5.1+cu121'}
PyTorch has not been imported by this setup cell.


## 3. Optional CUDA PyTorch repair, then GPU verification

The command targets only this kernel's environment, using the [official CUDA 12.8 wheel index](https://download.pytorch.org/whl/cu128/torch/). It may change the installed PyTorch version; the pipeline requires >=2.5 and checks its numerical behavior before training. See the [official installer](https://pytorch.org/get-started/locally/) for other builds. CUDA correctness is tested on the selected device; no full training starts during setup.


In [ ]:
INSTALL_CUDA_PYTORCH = True  # True once if your installed torch is CPU-only.
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu128"
if INSTALL_CUDA_PYTORCH:
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch is already loaded. Restart the kernel, then run only cells 3, 5 and this repair cell before the GPU cell.")
    command = [str(PYTHON), "-m", "pip", "install", "--upgrade", "--force-reinstall",
               "torch>=2.5", "--index-url", TORCH_INDEX_URL]
    print(subprocess.list2cmdline(command))
    subprocess.run(command, check=True, cwd=REPO_ROOT)
    raise RuntimeError("Installation finished. Restart the kernel, set INSTALL_CUDA_PYTORCH=False, then rerun from cell 3.")
print("Existing environment retained; checking CUDA next.")


Existing environment retained; checking CUDA next.


In [4]:
import numpy as np
import pandas as pd
import scipy
import torch

version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
assert version >= (2, 5), f"PyTorch >=2.5 required; found {torch.__version__}."
print({"python": str(PYTHON), "torch": str(torch.__version__), "cuda_build": torch.version.cuda})
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this kernel. If torch is CPU-only, run the optional CUDA repair cell, "
                       "restart the kernel, then rerun from cell 3. Training will not fall back to CPU.")
GPU_INDEX = 0
# The CLI is a separate process, so select the same device there explicitly.
CUDA_DEVICE_SELECTION = os.environ.get("CUDA_VISIBLE_DEVICES", str(GPU_INDEX))
torch.cuda.set_device(GPU_INDEX)
properties = torch.cuda.get_device_properties(GPU_INDEX)
print({"gpu": properties.name, "memory_GiB": round(properties.total_memory / 2**30, 2),
       "compute_capability": (properties.major, properties.minor)})
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
# A real CUDA + FP64 backward check before the more extensive trainer preflight.
with torch.enable_grad():
    probe = torch.tensor([[2., .1], [.1, 1.]], device="cuda", dtype=torch.float64, requires_grad=True)
    torch.linalg.solve(probe, torch.ones(2, 1, device="cuda", dtype=torch.float64)).square().sum().backward()
    assert torch.isfinite(probe.grad).all().item()
torch.cuda.synchronize()
del probe
torch.cuda.empty_cache()
print("CUDA allocation, FP64 solve and backward passed.")


{'python': 'A:\\Projects\\ARES\\.venv\\Scripts\\python.exe', 'torch': '2.5.1+cu121', 'cuda_build': '12.1'}
{'gpu': 'NVIDIA GeForce RTX 4080 Laptop GPU', 'memory_GiB': 11.99, 'compute_capability': (8, 9)}
CUDA allocation, FP64 solve and backward passed.


## 4. Choose local output and resume settings

Results are saved on your local disk under `results/augmented_saits_barrier/<RUN_NAME>`. No Google Drive or Colab service is needed. Keep the machine awake and VS Code's kernel running while training. A kernel interruption stops the training child process; resuming restores the latest completed epoch.

Set `RESUME=True` to continue an existing experiment with unchanged configuration. After a test reservation exists, this notebook resumes evaluation/export only. Use a new run name for changed model, splits, batch sizes, or optimization settings. The local configuration is saved beside the run, keeping a new experiment directory empty until the trainer creates it.


In [5]:
RUN_NAME = "ares_augmented_saits_barrier_local_seed42_v1"
RESUME = False
OUTPUT_ROOT = REPO_ROOT / "results" / "augmented_saits_barrier"
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
assert all(character not in RUN_NAME for character in r'<>:"/\|?*'), "Use a Windows-safe run name."
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUTPUT_ROOT / RUN_NAME
CONFIG = OUTPUT_ROOT / (RUN_NAME + ".config.json")
if OUT.exists() and any(OUT.iterdir()) and not RESUME:
    raise FileExistsError("Existing experiment: set RESUME=True or choose a new RUN_NAME.")
print("Checkpoints and evaluation:", OUT)
print("Local configuration:", CONFIG)
print("Free output disk space (GiB):", round(shutil.disk_usage(OUTPUT_ROOT).free / 2**30, 1))


Checkpoints and evaluation: A:\Projects\ARES\results\augmented_saits_barrier\ares_augmented_saits_barrier_local_seed42_v1
Local configuration: A:\Projects\ARES\results\augmented_saits_barrier\ares_augmented_saits_barrier_local_seed42_v1.config.json
Free output disk space (GiB): 199.6


## 5. Validate the MATLAB plant calibration in this runtime

This reads immutable MATLAB derivative, Jacobian, allocation, and 0.1 s trajectory fixtures. It regenerates sibling `validation.json` tied to the calibration, fixtures, and normalized Python source hashes. Stage 2 refuses a failed/missing/stale report. Do not change the immutable calibration or invent hardware error bounds.


In [6]:
subprocess.run([
    str(PYTHON), str(REPO_ROOT / "scripts/validate_dynamics_calibration.py"),
    "--calibration", str(CALIBRATION),
], check=True, cwd=REPO_ROOT)
from validate_dynamics_calibration import verify_validation_report
parity = verify_validation_report(CALIBRATION)
assert parity["passed"]
print({"passed": parity["passed"], "plant_step_s": parity["integration_dt_s"],
       "gate_groups": list(parity["gates"]),
       "calibration_sha256": parity["calibration_sha256"]})
calibration = json.loads(CALIBRATION.read_text(encoding="utf-8-sig"))
print("Measured horizontal tracking bound (m/s^2):", calibration["tracking_error"]["bound_mps2"])
print("Bounds are empirical software-domain measurements, not hardware certificates.")


{'passed': True, 'plant_step_s': 0.001, 'gate_groups': ['derivative_float64', 'derivative_float32', 'acceleration_allocation_float64', 'continuous_control_jacobian', 'zoh_autograd_vs_central_difference', 'trajectory_float64', 'endpoint_float64', 'collision_trace_positions', 'trace_endpoint_state', 'half_step_convergence', 'trajectory_float32', 'allocation_design_bounds'], 'calibration_sha256': '6693f052de9ccb181f989c8a47fb7c97ab777a34c89b80f269d7df0f48bf9c0b'}
Measured horizontal tracking bound (m/s^2): 0.21924911456604623
Bounds are empirical software-domain measurements, not hardware certificates.


## 6. Save local GPU settings and verify all 80 episodes

The full experiment retains 120 reconstruction epochs, 30 fine-tuning epochs, 200-step training rollouts, and 600-step validation/test rollouts. Neural work uses CUDA FP16 mixed precision; the constrained QP uses FP64. On this 12 GB machine, rollout batch size starts at four and reconstruction batch size at 128. The exact settings are persisted before fitting.

Plant compilation is disabled on native Windows for this local configuration; the same 1 ms calibrated plant still executes on CUDA. This avoids requiring a separate compiler toolchain. No rollout horizon, dataset scope, or safety constraint is reduced. Initial admissibility and rejected-run reporting are retained.


In [7]:
from ares_safe_training.config import PipelineConfig

configuration = json.loads(BASE_CONFIG.read_text(encoding="utf-8"))
configuration.update(device="cuda", amp=True, pretrain_batch_size=128,
                     rollout_batch_size=4, compile_dynamics=(os.name != "nt"))
configuration = PipelineConfig.from_dict(configuration).to_dict()
if CONFIG.exists():
    saved_configuration = json.loads(CONFIG.read_text(encoding="utf-8"))
    if saved_configuration != configuration:
        raise ValueError("The saved local configuration differs. Keep the original settings for resume, "
                         "or choose a new RUN_NAME before changing settings.")
else:
    with CONFIG.open("x", encoding="utf-8") as stream:
        json.dump(configuration, stream, indent=2, allow_nan=False)
        stream.write("\n")
print(json.dumps(configuration, indent=2))
from colab_episode_starter import EpisodeDataset  # Shared local loader; no Colab dependency.
index_only = EpisodeDataset(REPO_ROOT, BATCH_ID, verify_hashes=True, expected_runs=80)
assert len(index_only.runs) == 80
print("Recorded scans:", int(index_only.runs["ScanCount"].sum()))
print("Axis counts:", index_only.runs["Axis"].value_counts().to_dict())
print("Recorded collisions:", int(index_only.runs["CollisionOccurred"].sum()))
del index_only


{
  "seed": 42,
  "device": "cuda",
  "amp": true,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calibration_fraction_of_train": 0.2,
  "model": {
    "window_size": 50,
    "d_model": 64,
    "n_heads": 4,
    "d_ff": 128,
    "temporal_layers": 1,
    "state_layers": 2,
    "dropout": 0.1
  },
  "pretrain_epochs": 120,
  "pretrain_batch_size": 128,
  "pretrain_learning_rate": 0.001,
  "finetune_epochs": 30,
  "rollout_batch_size": 4,
  "rollout_steps": 200,
  "evaluation_steps": 600,
  "gradient_horizon": 10,
  "control_dt": 0.1,
  "plant_dt": 0.001,
  "finetune_learning_rate": 0.0001,
  "barrier_learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "patience": 20,
  "clip_grad_norm": 1.0,
  "observed_reconstruction_weight": 0.1,
  "auxiliary_reconstruction_weight": 0.1,
  "tracking_weight": 0.1,
  "control_correction_weight": 0.05,
  "barrier_violation_weight": 1.0,
  "geometry_violation_weight": 5.0,
  "tracking_slack_weight": 0.01,
  "reconstruction_weight": 1.0,
  "uncertaint

## 7. Run both training stages, sealed test evaluation, and export

The first rollout scan and initial 5 s are measured. Later hidden sectors are reconstructed; the physical state evolves from issued commands, not recorded future states. The new frozen goal/sector controller, learned-gain hard QP, exact allocation, and calibrated 13-state plant are used together. Geometry is restricted to simulator sensor generation, supervision, and evaluation.

The log is stored **beside** the output directory so that a new output remains empty until the CLI creates its manifest. `--resume` resumes completed checkpoints and preserves source/data/configuration checks.


In [ ]:
from datetime import datetime, timezone

test_reserved = any((OUT / name).exists() for name in ("test_evaluation_intent.json", "test_evaluation.json"))
if test_reserved and not RESUME:
    raise RuntimeError("Test access is reserved. Set RESUME=True to recover the same selected evaluation/export.")
STAGE = "evaluate" if test_reserved else "all"
command = [str(PYTHON), "-u", str(REPO_ROOT / "scripts/train_augmented_saits_barrier.py"),
           "--repo-root", str(REPO_ROOT), "--output-dir", str(OUT),
           "--calibration", str(CALIBRATION), "--config", str(CONFIG),
           "--stage", STAGE, "--device", "cuda"]
if RESUME:
    command.append("--resume")
TRAIN_ENV = dict(os.environ, PYTHONUTF8="1", PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1",
                 CUDA_VISIBLE_DEVICES=CUDA_DEVICE_SELECTION)
log_dir = OUTPUT_ROOT / "logs"
log_dir.mkdir(exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
log_path = log_dir / f"{RUN_NAME}.{stamp}.log"
print("Stage:", STAGE, "| Interpreter:", PYTHON)
print("Console log:", log_path)
with log_path.open("x", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=REPO_ROOT, env=TRAIN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    print("Training process ID:", process.pid)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Training stopped. Resume the same run from its last completed epoch with RESUME=True.")
        raise
    finally:
        process.stdout.close()
if return_code != 0:
    raise RuntimeError(f"Training exited with code {return_code}. Inspect {log_path} before resuming.")
print("Training/evaluation command completed. Review the held-out results below.")


Stage: all | Interpreter: A:\Projects\ARES\.venv\Scripts\python.exe
Console log: A:\Projects\ARES\results\augmented_saits_barrier\logs\ares_augmented_saits_barrier_local_seed42_v1.20261110T030540_583382Z.log
Training process ID: 31344
Outer routes: train=56, validation=12, test=12. Fit/calibration inside train=44/12.
Device=cuda, AMP FP16=True; real pretraining windows=20470; no online map input.
Stage 1 epoch 1: loss=0.11596, validation=0.11257
Stage 1 epoch 2: loss=0.07741, validation=0.09758
Stage 1 epoch 3: loss=0.05993, validation=0.08685
Stage 1 epoch 4: loss=0.04783, validation=0.08366
Stage 1 epoch 5: loss=0.04210, validation=0.09336
Stage 1 epoch 6: loss=0.03804, validation=0.08713
Stage 1 epoch 7: loss=0.03493, validation=0.08444
Stage 1 epoch 8: loss=0.03206, validation=0.08169
Stage 1 epoch 9: loss=0.02985, validation=0.08543
Stage 1 epoch 10: loss=0.02752, validation=0.08422
Stage 1 epoch 11: loss=0.02739, validation=0.08300
Stage 1 epoch 12: loss=0.02513, validation=0.082

## 8. Read the selected test result and exposure/failure details

Small reconstruction error or a completed epoch does not establish navigation safety. Inspect actual outage exposure, filter failures, collisions, clearance, altitude, termination reasons, and executed duration. A route that failed before its first outage provides no evidence of reconstruction-driven control during failure. Before the first test windows are created, `test_evaluation_intent.json` reserves access for one selected checkpoint and closes fitting, including after an interruption. `test_evaluation.json` seals the completed result. Resume only the same checkpoint with `--stage evaluate --resume`; sealed-result recovery completes export/summary without repeating test evaluation.


In [ ]:
experiment = json.loads((OUT / "experiment.json").read_text())
print("Outer split sizes:", {key: len(value) for key, value in experiment["outer_splits"].items()})
print("Gradient-fitting runs:", len(experiment["training_run_indices"]))
print("Empirical-calibration runs within train:", len(experiment["calibration_run_indices_within_train"]))
assert len(experiment["training_run_indices"]) == 44
assert len(experiment["calibration_run_indices_within_train"]) == 12
result_path = OUT / "summary.json"
if not result_path.is_file():
    result_path = OUT / "test_evaluation.json"
result = json.loads(result_path.read_text())
print(json.dumps(result, indent=2))
closed_path = OUT / "evaluation/test_closed_loop.json"
if closed_path.is_file():
    closed = json.loads(closed_path.read_text())
    records = closed.get("records", [])
    if records:
        display(pd.DataFrame(records))
print("This result does not certify safety or establish transfer to another obstacle layout.")


## 9. Retain the later-integration package

The `all` workflow normally creates `OUT/deployment`. The recovery below uses `--stage evaluate --resume` if publication is incomplete. A completed sealed result is reused without re-evaluating the test set; the selected checkpoint and experiment must stay unchanged. Existing export destinations are never overwritten. The deployment bundle excludes the obstacle map and optimizer state. Optional ONNX contains reconstruction only, not the online QP.

The older single-episode MATLAB MEX **cannot load this model**. Future Simulink work must port and parity-check both encoders, causal features/masks/anchors, the new frozen controller/reference, gain network, online QP, allocation, and timing.


In [ ]:
DEPLOYMENT = OUT / "deployment"
if not DEPLOYMENT.exists() or not (OUT / "summary.json").is_file():
    assert any((OUT / name).is_file() for name in ("test_evaluation_intent.json", "test_evaluation.json")), "No test reservation exists to recover."
    subprocess.run([str(PYTHON), str(REPO_ROOT / "scripts/train_augmented_saits_barrier.py"),
        "--repo-root", str(REPO_ROOT), "--output-dir", str(OUT), "--calibration", str(CALIBRATION),
        "--config", str(CONFIG), "--stage", "evaluate", "--resume", "--device", "cuda"],
        check=True, cwd=REPO_ROOT, env=TRAIN_ENV)
manifest = json.loads((DEPLOYMENT / "deployment.json").read_text(encoding="utf-8"))
import hashlib, zipfile
for name, artifact in manifest["artifacts"].items():
    assert Path(name).name == name, "Invalid artifact filename."
    assert hashlib.sha256((DEPLOYMENT / name).read_bytes()).hexdigest() == artifact["sha256"], f"Artifact mismatch: {name}"
checkpoint_id = manifest["provenance"]["source_checkpoint_sha256"][:12]
archive_base = OUTPUT_ROOT / f"{RUN_NAME}_deployment_{checkpoint_id}"
archive_path = Path(str(archive_base) + ".zip")
if not archive_path.exists():
    shutil.make_archive(str(archive_base), "zip", root_dir=DEPLOYMENT)
# Re-running this cell safely reuses the ZIP only when its contents still match.
with zipfile.ZipFile(archive_path) as saved:
    for path in DEPLOYMENT.iterdir():
        if path.is_file():
            assert hashlib.sha256(saved.read(path.name)).digest() == hashlib.sha256(path.read_bytes()).digest(), f"Stale ZIP: {path.name}"
print("Deployment:", DEPLOYMENT)
print("Local integration ZIP:", archive_path)
print("Also retain:", OUT / "stage2/best.pt", "and the experiment/evaluation summaries.")
display(FileLink(os.path.relpath(archive_path, Path.cwd())))


## Optional: staged execution and reconstruction-only ONNX

For an intentional staged run, use the same CLI and output: `--stage pretrain`, then `--stage finetune --resume`, then `--stage evaluate --resume`. Before test reservation, resume does not authorize changes to immutable settings. Once test access is reserved, use only `--stage evaluate --resume` for the same selected weights; do not resume fitting or extend epochs. After selection, `scripts/export_augmented_saits_barrier.py --checkpoint OUT/stage2/best.pt --output-dir NEW --onnx` adds optional batch-one reconstruction ONNX if the `onnx` package is installed. The hard QP, nominal controller, and allocation remain separate required online components.

Foundations: [SAITS](https://arxiv.org/abs/2202.08516), [high-order CBFs](https://calinbelta.com/wp-content/uploads/2023/12/High-Order_Control_Barrier_Functions.pdf), and [BarrierNet](https://arxiv.org/abs/2111.11277). The project-specific empirical sensing and dynamics assumptions remain essential.

Local CPU checks and a one-epoch reduced-horizon workflow on all 80 actual episodes passed through both stages, test sealing, and export; CUDA-specific execution was skipped locally. This is an execution check, not trained-performance or outage-control evidence. The full local GPU run remains necessary.

Local outputs remain under `results/augmented_saits_barrier/`. They are ignored by Git; back up selected checkpoints and the deployment ZIP separately. Restart VS Code's kernel after changing installed PyTorch packages.
